# Clase 182 — Intervalos de confianza

Un IC95 % **no** significa "95 % de probabilidad de que el parámetro esté aquí", sino que el 95 % de los intervalos construidos así contendrían el parámetro. Construimos IC para media (t, bootstrap) y proporción (Wald, Wilson, Clopper-Pearson).

Requiere: `numpy`, `scipy`, `statsmodels`, `matplotlib`.

## 1. IC t para la media

`x̄ ± t_{α/2, n-1}·(s/√n)`. La t tiene colas más anchas que la normal, lo que ensancha correctamente el IC con `n` chico.

In [ ]:
import numpy as np
from scipy import stats
from statsmodels.stats.proportion import proportion_confint
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

x = rng.normal(20, 8, 200)
n = len(x)
mean, sem = x.mean(), stats.sem(x)
lo, hi = stats.t.interval(0.95, n - 1, loc=mean, scale=sem)
print(f"media={mean:.2f}  IC95% t = ({lo:.2f}, {hi:.2f})")
assert lo < mean < hi

## 2. IC para proporción extrema

Con `p̂` cerca de 0, **Wald** puede dar un límite inferior negativo (imposible). Wilson y Clopper-Pearson (`beta`) se mantienen dentro de [0, 1].

In [ ]:
count = int(rng.binomial(100, 0.03))
nn = 100
print(f"count={count}  n={nn}  p̂={count/nn:.3f}")
for method in ("normal", "wilson", "beta"):
    l, h = proportion_confint(count, nn, alpha=0.05, method=method)
    tag = "  <- límite inferior imposible (<0)" if l < 0 else ""
    print(f"  {method:8s}: ({l:.4f}, {h:.4f})  ancho={h-l:.4f}{tag}")
wl, _ = proportion_confint(count, nn, method="normal")
assert wl < 0.05

## 3. Cobertura empírica

Construimos 5 000 IC95 % t sobre muestras de `N(50,10)` y contamos qué fracción contiene el verdadero `μ=50`. Debe ser ≈ 0.95.

In [ ]:
mu_true = 50.0
reps = 5000
hits = 0
for _ in range(reps):
    s = rng.normal(mu_true, 10, 30)
    l, h = stats.t.interval(0.95, 29, loc=s.mean(), scale=stats.sem(s))
    hits += (l <= mu_true <= h)
cov = hits / reps
print(f"cobertura empírica IC95% t = {cov:.3f} (nominal 0.95)")
assert 0.93 < cov < 0.97

## 4. IC bootstrap (sin supuestos paramétricos)

`scipy.stats.bootstrap` resamplea con reemplazo. Con datos ~normales coincide con el IC t.

In [ ]:
boot = stats.bootstrap((x,), np.mean, n_resamples=10_000,
                       method="percentile", random_state=rng)
print(f"IC95% bootstrap = ({boot.confidence_interval.low:.2f}, {boot.confidence_interval.high:.2f})")
print(f"IC95% t         = ({lo:.2f}, {hi:.2f})")

## 5. Tamaño de muestra y margen de error

Para un margen de error `ME`, `n = (z·√(p(1-p))/ME)²`. El peor caso es `p=0.5`. El ancho del IC decrece como `1/√n`.

In [ ]:
z = stats.norm.ppf(0.975)
ME = 0.03
p_worst = 0.5
n_req = (z ** 2 * p_worst * (1 - p_worst)) / ME ** 2
print(f"n requerido (ME=±3%, 95%, p=0.5) = {np.ceil(n_req):.0f}")
assert 1000 < n_req < 1100

ns = np.arange(20, 2000, 20)
widths = 2 * z * np.sqrt(0.25 / ns)
plt.figure(figsize=(6, 4))
plt.plot(ns, widths)
plt.axhline(2 * ME, color="r", ls="--", label="objetivo ±3%")
plt.xlabel("n"); plt.ylabel("ancho del IC"); plt.legend()
plt.title("El ancho del IC decrece como 1/√n")
plt.tight_layout(); plt.show()

## Ejercicios

1. Simulá `n=500` respuestas con `p=0.78`; construí los 3 IC de proporción y compará sus anchos.
2. Repetí la cobertura empírica con `n=8` (datos asimétricos): mostrá que el IC t deja de cubrir el 95 % nominal.
3. Verificá la relación IC ↔ test: si el IC95 % de `μ_a - μ_b` excluye 0, el t-test bilateral rechaza `H₀`.

## Conclusiones

- El IC es una propiedad del **procedimiento**, no del intervalo puntual; el parámetro es fijo.
- Para la media usá siempre t (con `n` grande coincide con z).
- Para proporciones, **Wilson** por default; Wald falla con `p` extrema o `n` chico.
- El bootstrap libera de supuestos paramétricos; el margen de error decrece con `√n` (4× datos para la mitad de ancho).